# admin

> Workspace user and license administration through Google APIs.

In [ ]:
#| default_exp admin

In [ ]:
#| export
from fastcore.utils import *
from fastgws.core import GWSApi

`WorkspaceAdmin` keeps the Admin Directory and Enterprise License Manager clients together. User creation, licensing, suspension, and deletion remain separate operations, so callers choose the lifecycle they need.

## Users

Construct the administrator from credentials that include `admin.directory.user` and `apps.licensing`. `create_user` mirrors Google’s required identity fields while keeping the password and organizational policy explicit.

In [ ]:
#| export
class WorkspaceAdmin:
    "High-level Workspace user and license administration"
    def __init__(self, creds):
        self.directory = GWSApi('admin', version='directory_v1', creds=creds)
        self.licensing = GWSApi('licensing', version='v1', creds=creds)

    async def create_user(
        self,
        primary_email:str, # New user's primary email address
        given_name:str, # Given name
        family_name:str, # Family name
        password:str, # Initial password
        org_unit_path:str='/', # Parent organizational unit
        change_password_at_next_login:bool=False, # Force a password change on first login?
        **kwargs
    ):
        "Create a Workspace user"
        return await self.directory.users.insert(primary_email=primary_email,
            name=dict(given_name=given_name, family_name=family_name), password=password,
            org_unit_path=org_unit_path, change_password_at_next_login=change_password_at_next_login, **kwargs)

## Licences and lifecycle

Licensing is deliberately independent from creation because a domain can auto-assign a licence by organizational unit. `assign_license` and `remove_license` use explicit product and SKU IDs; suspension and deletion only change the Directory user.

In [ ]:
#| export
@patch
async def assign_license(
    self:WorkspaceAdmin,
    user_id:str, # User's primary email address
    sku_id:str, # Workspace product SKU
    product_id:str='Google-Apps', # Workspace product ID
):
    "Assign a product licence to a user"
    return await self.licensing.license_assignments.insert(product_id=product_id, sku_id=sku_id, user_id=user_id)


`remove_license` revokes one explicit product/SKU assignment without changing the Directory user.


In [ ]:
@patch
async def remove_license(
    self:WorkspaceAdmin,
    user_id:str, # User's primary email address
    sku_id:str, # Workspace product SKU
    product_id:str='Google-Apps', # Workspace product ID
):
    "Remove a product licence from a user"
    return await self.licensing.license_assignments.delete(product_id=product_id, sku_id=sku_id, user_id=user_id)


`suspend_user` can suspend or restore an account independently of its licences.


In [ ]:
@patch
async def suspend_user(
    self:WorkspaceAdmin,
    user_key:str, # Primary email, alias, or immutable user ID
    suspended:bool=True, # Suspend rather than restore the user?
):
    "Set a user's suspension state"
    return await self.directory.users.update(user_key=user_key, suspended=suspended)


`delete_user` permanently removes the Directory user and remains a separate, explicit operation.


In [ ]:
@patch
async def delete_user(
    self:WorkspaceAdmin,
    user_key:str, # Primary email, alias, or immutable user ID
):
    "Delete a Workspace user"
    return await self.directory.users.delete(user_key=user_key)


A complete provisioning sequence stays explicit:

```python
scopes = ['https://www.googleapis.com/auth/admin.directory.user',
          'https://www.googleapis.com/auth/apps.licensing']
creds = await oauth_creds(account='admin@example.com', scopes=scopes)
admin = WorkspaceAdmin(creds)
user = await admin.create_user('new@example.com', 'New', 'User', password,
                               org_unit_path='/Internal')
await admin.assign_license(user.primaryEmail, 'workspace-sku-id')
```

If the organizational unit auto-licenses new users, inspect the assignment first and omit the final call.